In [27]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from xgboost import XGBClassifier

In [28]:
df = pd.read_csv('../After_EDA_data/Light_text.csv')

In [29]:
df

,uid,profile,anime_uid,score,scores,text
0,255938,DesolatePsyche,34096,8,"{'Overall': '8', 'Story': '8', 'Animation': '8...",first things first my reviews system is explai...
1,259117,baekbeans,34599,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",let me start off by saying that made in abyss ...
2,253664,skrn,28891,7,"{'Overall': '7', 'Story': '7', 'Animation': '9...",10 it is great especially the actions during t...
3,8254,edgewalker00,2904,9,"{'Overall': '9', 'Story': '9', 'Animation': '9...",story taking place 1 yr from where season 1 tr...
4,291149,aManOfCulture99,4181,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",kyoto animations greatest strength is being ab...
...,...,...,...,...,...,...
192107,240067,Unicorn819,1281,9,"{'Overall': '9', 'Story': '5', 'Animation': '1...",ok this anime is pretty old but here s the bac...
192108,285777,ShizzoSVH,1281,9,"{'Overall': '9', 'Story': '7', 'Animation': '9...",the dub for this anime is made this anime a fu...
192109,286904,AlluMan96,1281,3,"{'Overall': '3', 'Story': '3', 'Animation': '1...",some might argue that doing a review of a show...
192110,287903,AgentK300,1281,10,"{'Overall': '10', 'Story': '3', 'Animation': '...",absolutely hilarious i accidentally came acros...


In [30]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 192112 entries, 0 to 192111
Data columns (total 6 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   uid        192112 non-null  int64 
 1   profile    192112 non-null  object
 2   anime_uid  192112 non-null  int64 
 3   score      192112 non-null  int64 
 4   scores     192112 non-null  object
 5   text       192112 non-null  object
dtypes: int64(3), object(3)
memory usage: 8.8+ MB


In [31]:
df.shape

(192112, 6)

In [32]:
def classification(text):
    if text >= 5:
        return 'Good'
    return 'Bad'

In [33]:
df['target'] = df['score'].apply(classification)

In [34]:
df

,uid,profile,anime_uid,score,scores,text,target
0,255938,DesolatePsyche,34096,8,"{'Overall': '8', 'Story': '8', 'Animation': '8...",first things first my reviews system is explai...,Good
1,259117,baekbeans,34599,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",let me start off by saying that made in abyss ...,Good
2,253664,skrn,28891,7,"{'Overall': '7', 'Story': '7', 'Animation': '9...",10 it is great especially the actions during t...,Good
3,8254,edgewalker00,2904,9,"{'Overall': '9', 'Story': '9', 'Animation': '9...",story taking place 1 yr from where season 1 tr...,Good
4,291149,aManOfCulture99,4181,10,"{'Overall': '10', 'Story': '10', 'Animation': ...",kyoto animations greatest strength is being ab...,Good
...,...,...,...,...,...,...,...
192107,240067,Unicorn819,1281,9,"{'Overall': '9', 'Story': '5', 'Animation': '1...",ok this anime is pretty old but here s the bac...,Good
192108,285777,ShizzoSVH,1281,9,"{'Overall': '9', 'Story': '7', 'Animation': '9...",the dub for this anime is made this anime a fu...,Good
192109,286904,AlluMan96,1281,3,"{'Overall': '3', 'Story': '3', 'Animation': '1...",some might argue that doing a review of a show...,Bad
192110,287903,AgentK300,1281,10,"{'Overall': '10', 'Story': '3', 'Animation': '...",absolutely hilarious i accidentally came acros...,Good


In [35]:
df['target'].value_counts()

target
Good    169994
Bad      22118
Name: count, dtype: int64

In [36]:
df_majority = df[df['target'] == 'Good']
df_minority = df[df['target'] == 'Bad']

In [37]:
print(f"Shape majority:: {len(df_majority)}")
print(f"Shape minority:: {len(df_minority)}")

Shape majority:: 169994
Shape minority:: 22118


In [38]:
df_reduced = df_majority.sample(n=len(df_minority), random_state=42)

In [39]:
df_balanced = pd.concat([df_reduced, df_minority]).sample(frac=1, random_state=42)

In [40]:
df_balanced

,uid,profile,anime_uid,score,scores,text,target
90145,65641,rachel-chanx3,11739,9,"{'Overall': '9', 'Story': '10', 'Animation': '...",we still can t imagine our futures or foresee ...,Good
147133,232715,Dead_or_H3ntai,12055,4,"{'Overall': '4', 'Story': '4', 'Animation': '5...",i ll be quick about this read the brandish man...,Bad
80111,177049,Valkqt,17729,3,"{'Overall': '3', 'Story': '2', 'Animation': '6...",be it for the dorky humour or the sporadic bou...,Bad
129819,254924,xgreeneyednekox,34501,4,"{'Overall': '4', 'Story': '3', 'Animation': '5...",in short entertaining but don t expect too muc...,Bad
176820,304469,Blood_Diver_A,37302,8,"{'Overall': '8', 'Story': '7', 'Animation': '6...",this review contains spoilers a seriously unde...,Good
...,...,...,...,...,...,...,...
115802,202516,justa333,4722,9,"{'Overall': '9', 'Story': '7', 'Animation': '5...",the reason i enjoyed this anime is because of ...,Good
173615,299148,Artrill,37450,5,"{'Overall': '5', 'Story': '5', 'Animation': '3...",5 0 10 a blas boy stands on a bridge overlooki...,Good
135172,160993,literaturenerd,1280,2,"{'Overall': '2', 'Story': '2', 'Animation': '2...",overview we have enough people on this site re...,Bad
114811,207548,Xembled,30948,7,"{'Overall': '7', 'Story': '7', 'Animation': '7...",i know a lot of people don t really enjoy this...,Good


In [41]:
df_balanced = df_balanced.reset_index(drop=True)

In [42]:
df_balanced

,uid,profile,anime_uid,score,scores,text,target
0,65641,rachel-chanx3,11739,9,"{'Overall': '9', 'Story': '10', 'Animation': '...",we still can t imagine our futures or foresee ...,Good
1,232715,Dead_or_H3ntai,12055,4,"{'Overall': '4', 'Story': '4', 'Animation': '5...",i ll be quick about this read the brandish man...,Bad
2,177049,Valkqt,17729,3,"{'Overall': '3', 'Story': '2', 'Animation': '6...",be it for the dorky humour or the sporadic bou...,Bad
3,254924,xgreeneyednekox,34501,4,"{'Overall': '4', 'Story': '3', 'Animation': '5...",in short entertaining but don t expect too muc...,Bad
4,304469,Blood_Diver_A,37302,8,"{'Overall': '8', 'Story': '7', 'Animation': '6...",this review contains spoilers a seriously unde...,Good
...,...,...,...,...,...,...,...
44231,202516,justa333,4722,9,"{'Overall': '9', 'Story': '7', 'Animation': '5...",the reason i enjoyed this anime is because of ...,Good
44232,299148,Artrill,37450,5,"{'Overall': '5', 'Story': '5', 'Animation': '3...",5 0 10 a blas boy stands on a bridge overlooki...,Good
44233,160993,literaturenerd,1280,2,"{'Overall': '2', 'Story': '2', 'Animation': '2...",overview we have enough people on this site re...,Bad
44234,207548,Xembled,30948,7,"{'Overall': '7', 'Story': '7', 'Animation': '7...",i know a lot of people don t really enjoy this...,Good


In [43]:
x = df_balanced['text']
y = df_balanced['target']

In [44]:
le = LabelEncoder()

y_encoded = le.fit_transform(y)

In [45]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y_encoded, test_size=0.3, stratify=y_encoded, random_state=42
)

In [46]:
tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

In [47]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [48]:
model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=200,
    learning_rate=0.5
)

model.fit(X_train_tfidf, y_train)

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [49]:
y_pred = model.predict(X_test_tfidf)
y_prob = model.predict_proba(X_test_tfidf)[:,1]

In [50]:
print(classification_report(y_test, y_pred, target_names=le.classes_))

              precision    recall  f1-score   support

         Bad       0.87      0.87      0.87      6636
        Good       0.87      0.87      0.87      6635

    accuracy                           0.87     13271
   macro avg       0.87      0.87      0.87     13271
weighted avg       0.87      0.87      0.87     13271



In [51]:
print(confusion_matrix(y_test, y_pred))

[[5806  830]
 [ 861 5774]]


In [52]:
print(roc_auc_score(y_test, y_prob))

0.9406074650248717
